# WIP (2/2) — Constructor y contratos para ingesta de imágenes (Ungraph)

**Estado:** trabajo en progreso / diseño.

Notebook hermano: **`WIP_11_Image_Ingestion_ETI.ipynb`** (visión ETI).

## Tabla de componentes (propuesta)

| Tipo | Nombre | Responsabilidad |
|------|--------|-----------------|
| Value object | `ImageSourceDescriptor` | Ruta/URL, MIME, hash, dimensiones |
| ABC | `ImageToTextService` | `extract(descriptor) → {text, aux_metadata}` |
| Impl. | `VisionCaptionService` | Modelo multimodal / API cloud |
| Impl. | `OCRRasterService` | Tesseract, Azure DI, etc. |
| Impl. | `CompositeImageService` | Orquesta caption + OCR + fusión |
| Builder | `ImageDerivedDocumentBuilder` | Produce `Document` del dominio listo para `ChunkingService` |

## Criterios de aceptación (futuro)

- Tests unitarios con imágenes sintéticas y fixtures en `tests/`.
- Sin dependencias pesadas en el wheel base; extras explícitos en `pyproject.toml`.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Protocol


@dataclass
class ImageSourceDescriptor:
    path: Path
    source_id: str
    mime_type: str = "image/png"


class ImageToTextService(Protocol):
    """Contrato futuro: imagen → payload para construir Document/Chunk."""

    def extract(self, src: ImageSourceDescriptor) -> dict[str, Any]: ...


class ImageDerivedDocumentBuilder:
    """Orquesta extract + Document.create (cuando exista API estable)."""

    def __init__(self, extractor: ImageToTextService) -> None:
        self._extractor = extractor

    def build_payload(self, src: ImageSourceDescriptor) -> dict[str, Any]:
        return self._extractor.extract(src)
